In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
import requests

In [ ]:
# The user wants to convert USD to INR using tools.
# When the user gives this prompt, the system first gets the conversion factor using the get_conversion_factor tool.
# After obtaining the conversion factor, it then calls the convert tool to convert USD to INR using the conversion factor returned by the get_conversion_factor tool.

In [ ]:
#exchange rate api is used

In [ ]:
# Import decorator to convert Python functions into LangChain tools
from langchain_core.tools import tool

# Import helper used to automatically inject arguments between tools
from langchain_core.tools import InjectedToolArg

# Used for advanced type annotations
from typing import Annotated

# Library used to make HTTP API requests
import requests


# ---------------------------------------------------------
# TOOL 1: Fetch currency conversion rate
# ---------------------------------------------------------
@tool
def get_conversion_factor(base_currency: str, target_currency: str) -> float:
    """
    Fetch the currency conversion rate between
    a base currency and a target currency.

    Example:
    USD → INR
    """

    # Construct API URL dynamically using user inputs
    url = (
        f"https://v6.exchangerate-api.com/v6/"
        f"c754eab14ffab33112e380ca/pair/"
        f"{base_currency}/{target_currency}"
    )

    # Send GET request to exchange-rate server
    response = requests.get(url)

    # Convert API response into Python dictionary
    data = response.json()

    # Return only the conversion rate value
    return data["conversion_rate"]


# ---------------------------------------------------------
# TOOL 2: Convert currency value
# ---------------------------------------------------------
@tool
def convert(
    base_currency_value: int,
    # InjectedToolArg means this value is automatically
    # passed internally (not asked from the user)
    # Whatever conversion factor we receive from the first tool call will be stored in the conversion_rate variable — this will be controlled by the developer.
    conversion_rate: Annotated[float, InjectedToolArg]
) -> float:
    """
    Convert base currency amount into target currency
    using the provided conversion rate.
    """

    # Perform currency conversion calculation
    converted_value = base_currency_value * conversion_rate

    # Return converted amount
    return converted_value

In [ ]:
#Print the arguments used in the convert tool
convert.args

In [ ]:
#It will return the json which also contains the conversion factor
get_conversion_factor.invoke({'base_currency':'USD','target_currency':'INR'})

In [ ]:
#Invoke the convert tool with the base currency value and conversion rate
convert.invoke({'base_currency_value':10, 'conversion_rate':85.16})

In [ ]:
#create the llm
import os
from dotenv import load_dotenv      

load_dotenv("C:/Users/smi68/Desktop/My_Learning/Artificial-Intelligence/LangChain/LangChain_Code/.env")

# Create a connection to a Hugging Face hosted model (cloud-based)

llm = ChatOpenAI(
    model='mistralai/mistral-small-2506',
    api_key=os.getenv("LLM_API_KEY"),     # API key from .env
    base_url=os.getenv("LLM_BASE_URL")    # Custom provider endpoint
)

#Bind the llm with the tools
llm_with_tools = llm.bind_tools([get_conversion_factor, convert])

In [ ]:
query = "what is the conversion factor between INR and USD and convert 2 from INR to USD"
messages = [query]

In [ ]:
#we call with tool with the llm mesage
ai_message = llm_with_tools.invoke(messages)
#if we print we get the ai_message
ai_message
#print the tool calls
ai_message.too_calls

In [ ]:
messages.append(ai_message)

In [ ]:
import json

# Loop through all tool calls suggested by the AI message
for tool_call in ai_message.tool_calls:

    # -----------------------------------------------------
    # STEP 1: Execute first tool → get_conversion_factor
    # -----------------------------------------------------
    # This tool fetches the currency conversion rate
    if tool_call['name'] == 'get_conversion_factor':

        # Execute the tool using arguments provided by the LLM
        tool_message1 = get_conversion_factor.invoke(tool_call)

        #whatever we get tool_message1 is in strings formate which should be converted to json
        # Extract conversion rate from tool response (JSON format)
        conversion_rate = json.loads(
            tool_message1.content
        )['conversion_rate']

        # Store tool response in conversation history
        # so the agent knows this step already happened
        messages.append(tool_message1)

    # -----------------------------------------------------
    # STEP 2: Execute second tool → convert
    # -----------------------------------------------------
    # This tool performs the currency calculation
    if tool_call['name'] == 'convert':

        # Inject conversion_rate obtained from Tool 1
        # (developer-controlled value, not user input)
        tool_call['args']['conversion_rate'] = conversion_rate

        # Execute conversion tool with updated arguments
        tool_message2 = convert.invoke(tool_call)

        # Add result to messages history
        messages.append(tool_message2)

In [ ]:
#Messages will be having - Human message, ai message, tool_message1, tool_message1
messages

In [ ]:
#manually exceute the tool
llm_with_tools.invoke(messages).content

In [ ]:
from langchain.agents import initialize_agent, AgentType

# Step 5: Initialize the Agent ---
agent_executor = initialize_agent(
    tools=[get_conversion_factor, convert],
    llm=llm,
    agent=AgentType.STRUCTURED_CHAT_ZERO_SHOT_REACT_DESCRIPTION,  # using ReAct pattern
    verbose=True  # shows internal thinking
)

In [ ]:
# --- Step 6: Run the Agent ---
user_query = "Hi how are you?"

response = agent_executor.invoke({"input": user_query})